# LLM Fine-Tuning (gemma-2b + LoRA)

공개된 대규모 언어모델 **"gemma-2b"** 을 Foundation Model로 하고, LoRA 기법을 적용하여 효과적으로 파인튜닝하는 실습을 진행해 보겠습니다.  
- Task: Causal Language Model (Instruct Fine-Tuning)
- Foundation Model: “gemma-2b”
- Dataset: “KorAlpaca”
- Trainer: Huggingface PEFT/LoRA

## 0. Setup

In [ ]:
import os
os.environ['HF_HOME'] = 'D:/HF/cache'
os.environ['HF_DATASETS'] = 'D:/HF/datasets'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = "1"

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["FSDP_CPU_RAM_EFFICIENT_LOADING"] = "false"

In [ ]:
# !pip install -q transformers==4.51.1
# !pip install -q datasets==3.2.0
# !pip install -q peft==0.15.1
# !pip install -q trl==0.16.1
# !pip install -q accelerate==1.3.0

In [ ]:
# !pip install -q bitsandbytes

In [ ]:
from IPython.display import display
from tqdm.notebook import tqdm as notebook_tqdm

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [ ]:
"""
from huggingface_hub import notebook_login
notebook_login()
"""

In [ ]:
"""
from huggingface_hub import login
from dotenv import load_dotenv
from pathlib import Path
import os

dotenv_path = Path('Z:/Misc/access_token.env')
load_dotenv(dotenv_path=dotenv_path)
access_key=os.getenv('HF_TOKEN')

print(f"HF Access Key: {access_key}")
"""

In [ ]:
!nvidia-smi

## 1. DataSet: KoAlpaca v1.1a

질문을 전달하면 답변하는 형태로 대규모 언어모델을 파인튜닝하기 위한 학습 데이터셋을 준비하겠습니다.  
데이터셋은 **Instruction** (지시사항)과 **Output** (출력)의 쌍으로 구성되어 있습니다.  
KoAlpaca 데이터셋은 지식iN 기반의 질문-답변 데이터셋이며, Huggingface Dataset 에 공개되어 있습니다.

In [ ]:
from datasets import load_dataset

# dataset = load_dataset("beomi/KoAlpaca-v1.1a")
dataset = load_dataset("//swschoolavdazfiles002.file.core.windows.net/aias-language/Dataset/KoAlpaca-v1.1a")

dataset

In [ ]:
dataset['train'][0]

파인튜닝하기 위하여 학습 데이터를 다음과 같은 Instruction Format 형태로 변환을 합니다.  
```
### Question:

### Response: <eos>
```

In [ ]:
def generate_prompt(example):
    output_texts = []
    for i in range(len(example['instruction'])):
        # [실습] 다음 코드를 완성하세요!!
        # Example에 대해 위에서 정의한 Instruction Format 형태로 변환합니다.
        prompt = f"### Question: {example['instruction'][i]}\n\n### Response: {example['output'][i]}<eos>"
        output_texts.append(prompt)
    return output_texts

In [ ]:
train_data = dataset['train']
print(generate_prompt(train_data[:1])[0])

## 2. Foundation Model & Tokenizer

Foundation LLM 으로 "google/gemma-2b" 모델을 로딩합니다.  
- Lightweight, state-of-the-art open models
- Text-to-text, decoder-only LLM (English)
- Instruction following & Multi-turn conversations

In [ ]:
# BASE_MODEL = "google/gemma-2b"
BASE_MODEL = "//swschoolavdazfiles002.file.core.windows.net/aias-language/Model/gemma-2b"

# [실습] 다음 코드를 완성하세요!!
# 사전 학습된 'google/gemma-2b' 모델과 토크나이저를 가져옵니다.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = 'right'

In [ ]:
prompt = "건강하게 살기 위한 세 가지 방법은?"

In [ ]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)

# [실습] 다음 코드를 완성하세요!!
# Text-Generation Pipeline 관련 do_sample, temperature, top_k, top_p, repetition_penalty 등 파라미터를 설정합니다.
outputs = pipe(
    prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

print(outputs[0]["generated_text"][len(prompt):])

## 3. PEFT LoRA Setup

LoraConfig 함수를 통해 파인튜닝을 위한 주요 LoRA 파라미터를 설정합니다.

In [ ]:
# [실습] 다음 코드를 완성하세요!!
# LoRA 파인튜닝을 위한 Config를 설정합니다. r, lora_alpha, lora_dropout, target_modules, task_type
lora_config = LoraConfig(
    r=6,
    lora_alpha = 8,
    lora_dropout = 0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

## 4. Model Training (PEFT LoRA)

SFT Trainer 관련 필요한 학습 파라미터 설정한 후 PEFT LoRA 기반으로 파인튜닝을 위한 학습을 진행합니다.

In [ ]:
# [실습] 다음 코드를 완성하세요!!
# Training Arguments를 설정을 합니다. (num_train_epochs, max_steps,warmup_steps, etc.)
training_args = TrainingArguments(
    output_dir="D:/Trainer/PEFT-Trainer",
    num_train_epochs = 1,
    max_steps=1000, # 3000
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    label_names=["labels"],
    # optim="adamw_torch", # Default: Pytorch AdamW
    warmup_steps=10,
    learning_rate=2e-4,
    bf16=True, # fp16=True
    logging_steps=100,
    push_to_hub=False,
    report_to='none',
    dataloader_num_workers=2,
    dataloader_prefetch_factor=1,
)

In [ ]:
# [실습] 다음 코드를 완성하세요!!
# 파인튜닝을 위한 Trainer 설정을 합니다. (model, train_dataset, args, peft_config, formatting_func)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    peft_config=lora_config,
    formatting_func=generate_prompt,
    processing_class=tokenizer,
)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

LoRA 모델을 통해 학습 가능한 파라미터수는 전체 모델 파라미터(2,513,526,784)의 0.05% 수준입니다.

In [ ]:
print_trainable_parameters(model)

1000 스텝 학습에 10분 정도 소요되니, TrainingArguments에서 **`max_steps`** 설정하실 때 참고하시기 바랍니다.

In [ ]:
trainer.train()

학습된 LoRA Adapter 모델을 Local Directory 에 저장합니다.

In [ ]:
"""
ADAPTER_MODEL = "lora_adapter"

trainer.model.save_pretrained(ADAPTER_MODEL)
"""

## 5. Model Inference

파인튜닝된 모델을 이용하여 Generation 해 보도록 하겠습니다.

In [ ]:
pipe_finetuned = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)

In [ ]:
prompt = "건강하게 살기 위한 세 가지 방법은?"
formatted_prompt = f"### Question: {prompt}\n\n### Response:"

In [ ]:
outputs = pipe_finetuned(
    formatted_prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)
print(outputs[0]["generated_text"][len(formatted_prompt):])

- Ref. https://huggingface.co/docs/peft/main/en/task_guides/token-classification-lora